# Brain Mask Analysis - Segmentation Mask Volumetry

### Contact
- **Alexander R. Webber** (alexander.webber@fda.hhs.gov)
- **Seyed M. Kahaki** (seyed.kahaki@fda.hhs.gov)

---

Compares groups of NIfTI segmentation masks: it measures the physical volume of every mask, shows
center slices of a sample of cases side by side, and summarizes the per-group volume distributions.


## Input format

Groups are defined by glob patterns, so any directory layout works. A case is identified by the
directory names along its path, which is also how `excluded_cases` skips specific cases.

| Item | Details |
|---|---|
| **File type** | NIfTI (`.nii`, `.nii.gz`), any voxel grid and spacing |
| **Foreground** | Voxels `> 0`, except for the filenames listed in `inverted_mask_filenames` |
| **Volume** | `voxel count × voxel volume`, using the spacing in the NIfTI header, reported in cm³ |

### Conventional and inverted masks

Both mask families under these roots are handled by the same code path:

| Mask file | Meaning of positive voxels | Handling |
|---|---|---|
| `skull.nii.gz` | The skull itself - **foreground** | Voxels `> 0` are measured |
| `no_brain.nii.gz` | Everything that is *not* brain - **background** | Inverted automatically: voxels `<= 0` are measured |

Inversion is decided by filename through `inverted_mask_filenames`, which defaults to
`('no_brain.nii.gz',)`. Add any other background-encoded filename there.


## 1 · Imports


In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

from IPython.display     import display
from matplotlib          import pyplot as plt

from src.brain_pipeline  import run_mask_analysis


## 2 · Configuration

This is the only cell you need to edit. Each entry in `mask_analyses` becomes one independent
analysis with its own figures and tables - delete an entry to skip that mask family, or add one to
cover another.


In [ ]:
REAL_ROOT  = (
    '/projects01/didsr-aiml/yeelamelim.thompson/SDDT/brain/segmentation/'
    'segmentation/outputs/second_attempt/topcow/rsna_aneurysm'
)
SYNTH_ROOT = (
    '/projects01/didsr-aiml/yeelamelim.thompson/SDDT/brain/segmentation/'
    'segmentation/outputs/first_attempt/topcow/26_06_25_17_05_batch_100'
)

# One entry per mask family. Names are used in figure titles and output filenames.
mask_analyses = {
    'Brain Mask': {
        'input_groups': {
            'Real'     : [f'{REAL_ROOT}/*/no_brain.nii.gz'],
            'Synthetic': [f'{SYNTH_ROOT}/*/no_brain.nii.gz'],
        },
    },

    'Skull Mask': {
        'input_groups': {
            'Real'     : [f'{REAL_ROOT}/*/intermediate/total_segmentator/skull.nii.gz'],
            'Synthetic': [f'{SYNTH_ROOT}/*/intermediate/total_segmentator/skull.nii.gz'],
        },
    },
}

shared_settings = {
    'excluded_cases': {
        'Real': {
            # '1.2.826.0.1.3680043.8.498.87794163393266428648659243169230666286',
            # '1.2.826.0.1.3680043.8.498.10935907012185032169927418164924236382',
        },
    },

    'num_display'     : 5,     
    'seed'            : 42,    
    'inspect_extremes': True,

    # 'group_colors'           : {'Real': '#007CBA', 'Synthetic': '#FF8C42'},
    # 'inverted_mask_filenames': ('no_brain.nii.gz',),
    # 'save_outputs'           : False,
}


## 3 · Run

Each analysis writes its figures, CSV tables, and text summary to
`data/notebook_outputs/brain_masks/`, prefixed with its name.


In [ ]:
results = {
    name: run_mask_analysis(analysis_name=name, **settings, **shared_settings)
    for name, settings in mask_analyses.items()
}

## 4 · Results

- **Comparison grid** - center slices of the sampled cases, one row per group. Every panel of a
  given view shares one physical zoom window, so mask sizes can be compared directly across cards.
- **Volume distribution** - per-group histogram with a KDE curve, mean line, and shaded 95% CI.
- **Smallest-volume case** - a closer look at each group's smallest mask, which is where failed
  segmentations usually show up.

Summary columns: standard deviation and variance measure absolute spread; Q1/Q3/IQR describe the
middle 50% and resist outliers; CV expresses spread relative to the mean, which matters when groups
differ in average volume; skewness is positive for a longer right tail. Statistics needing more
cases than a group has are reported as `NA`.


In [ ]:
for name, analysis in results.items():
    for figure_name, fig in analysis['figures'].items():
        print(f'--- {name} · {figure_name} ---')
        display(fig)
        plt.close(fig)

    print(analysis['summary_text'])
    print()


In [ ]:
for name, analysis in results.items():
    print(f'--- {name} ---')
    display(analysis['case_statistics'].head(10))

    if analysis['errors']:
        print(f"{len(analysis['errors'])} mask(s) failed to load:")

        for path, message in analysis['errors']:
            print(f'  {path} - {message}')


## 5 · Saved outputs


In [ ]:
for name, analysis in results.items():
    for path in analysis['saved_paths']:
        print(path)
